In [7]:
# -------------------------------------------------
# STEP 1 — Import libraries
# -------------------------------------------------

import sqlite3
import pandas as pd
from pathlib import Path

# -------------------------------------------------
# STEP 2 — Connect to NordicFlow SQLite database
# -------------------------------------------------

db_path = Path("D:/NordicFlow Project - Full ERP project/"
    "Project-05-Python-ERP-Analytics/data/NordicFlow_ERP.db")

conn = sqlite3.connect(db_path)

# -------------------------------------------------
# STEP 3 — Load supplier and purchase-order data
# -------------------------------------------------

supplier = pd.read_sql_query(
    "SELECT * FROM supplier_master",
    conn
)

purchase_orders = pd.read_sql_query(
    "SELECT * FROM purchase_order_lines",
    conn
)

print("Suppliers:", len(supplier))
print("Purchase order lines:", len(purchase_orders))

# -------------------------------------------------
# STEP 4 — Prepare dates and numeric fields
# -------------------------------------------------

# Convert delivery dates to real datetime values
purchase_orders["Required_Delivery_Date"] = pd.to_datetime(
    purchase_orders["Required_Delivery_Date"],
    dayfirst=True,
    errors="coerce"
)

purchase_orders["Actual_Receipt_Date"] = pd.to_datetime(
    purchase_orders["Actual_Receipt_Date"],
    dayfirst=True,
    errors="coerce"
)

# Ensure price is numeric
purchase_orders["Unit_Price_EUR"] = pd.to_numeric(
    purchase_orders["Unit_Price_EUR"]
        .astype(str)
        .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

# Recalculate on-time delivery flag
purchase_orders["On_Time_Flag_Calc"] = (
    purchase_orders["Actual_Receipt_Date"]
    <= purchase_orders["Required_Delivery_Date"]
)

print(purchase_orders.dtypes)

# -------------------------------------------------
# STEP 5 — Build supplier performance KPIs
# -------------------------------------------------

# Calculate received spend at PO-line level
purchase_orders["Received_Spend_EUR"] = (
    purchase_orders["Received_Qty"]
    * purchase_orders["Unit_Price_EUR"]
)

# Calculate supplier-level performance
supplier_performance = (
    purchase_orders
    .groupby("Supplier_ID", as_index=False)
    .agg(
        PO_Lines=("PO_ID", "count"),
        Received_Qty=("Received_Qty", "sum"),
        Received_Spend_EUR=("Received_Spend_EUR", "sum"),
        Avg_Lead_Time_Days=("Actual_Lead_Time_Days", "mean"),
        Avg_Delivery_Delay_Days=("Delivery_Delay_Days", "mean"),
        Quality_Rejected_Qty=("Quality_Rejected_Qty", "sum"),
        On_Time_Delivery_Pct=("On_Time_Flag_Calc", "mean")
    )
)

# Convert OTD from decimal to percentage
supplier_performance["On_Time_Delivery_Pct"] = (
    supplier_performance["On_Time_Delivery_Pct"] * 100
)

supplier_performance.head()

# -------------------------------------------------
# STEP 6 — Add supplier master attributes
# -------------------------------------------------

supplier_scorecard = supplier_performance.merge(
    supplier[
        [
            "Supplier_ID",
            "Supplier_Name",
            "Supplier_Country",
            "Risk_Level",
            "Preferred_Status",
            "Target_Lead_Time_Days",
            "Target_On_Time_Rate"
        ]
    ],
    on="Supplier_ID",
    how="left"
)

supplier_scorecard.head()

# -------------------------------------------------
# STEP 7A — Convert supplier target OTD to numeric
# -------------------------------------------------

# Remove any % symbols or other text and convert to numeric
supplier_scorecard["Target_On_Time_Rate"] = pd.to_numeric(
    supplier_scorecard["Target_On_Time_Rate"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

# If source values are stored as 0.95, convert to 95
# If already stored as 95, keep as 95
supplier_scorecard["Target_On_Time_Pct"] = supplier_scorecard[
    "Target_On_Time_Rate"
].apply(
    lambda x: x * 100 if pd.notna(x) and x <= 1 else x
)

# Check the result
print(
    supplier_scorecard[
        ["Supplier_ID", "Target_On_Time_Rate", "Target_On_Time_Pct"]
    ]
)

# -------------------------------------------------
# STEP 7B — Calculate gap between actual and target OTD
# -------------------------------------------------

supplier_scorecard["OTD_Gap_Pct"] = (
    supplier_scorecard["On_Time_Delivery_Pct"]
    - supplier_scorecard["Target_On_Time_Pct"]
)

supplier_scorecard[
    [
        "Supplier_ID",
        "Supplier_Name",
        "On_Time_Delivery_Pct",
        "Target_On_Time_Pct",
        "OTD_Gap_Pct"
    ]
]

# -------------------------------------------------
# STEP 7 — Calculate quality and target performance
# -------------------------------------------------

# Quality rejection percentage
supplier_quality = (
    purchase_orders
    .groupby("Supplier_ID", as_index=False)
    .agg(
        Total_Received_Qty=("Received_Qty", "sum"),
        Total_Rejected_Qty=("Quality_Rejected_Qty", "sum")
    )
)

supplier_quality["Quality_Rejection_Pct"] = (
    supplier_quality["Total_Rejected_Qty"]
    / supplier_quality["Total_Received_Qty"]
    * 100
)

supplier_scorecard = supplier_scorecard.merge(
    supplier_quality[
        ["Supplier_ID", "Quality_Rejection_Pct"]
    ],
    on="Supplier_ID",
    how="left"
)

# Compare actual OTD against supplier target
supplier_scorecard["Target_On_Time_Pct"] = (
    supplier_scorecard["Target_On_Time_Rate"] * 100
)

supplier_scorecard["OTD_Gap_Pct"] = (
    supplier_scorecard["On_Time_Delivery_Pct"]
    - supplier_scorecard["Target_On_Time_Pct"]
)

supplier_scorecard.head()

# -------------------------------------------------
# STEP 8 — Create supplier review classification
# -------------------------------------------------

def classify_supplier(row):
    """
    Classify suppliers using delivery, quality
    and master-data risk indicators.
    """

    if (
        row["Risk_Level"] == "High"
        or row["On_Time_Delivery_Pct"] < 70
        or row["Quality_Rejection_Pct"] > 10
    ):
        return "High Review Priority"

    elif (
        row["Risk_Level"] == "Medium"
        or row["On_Time_Delivery_Pct"] < 90
        or row["Quality_Rejection_Pct"] > 5
    ):
        return "Medium Review Priority"

    else:
        return "Monitor"


supplier_scorecard["Review_Priority"] = (
    supplier_scorecard.apply(
        classify_supplier,
        axis=1
    )
)

supplier_scorecard[
    [
        "Supplier_ID",
        "Supplier_Name",
        "Risk_Level",
        "On_Time_Delivery_Pct",
        "Avg_Delivery_Delay_Days",
        "Quality_Rejection_Pct",
        "Review_Priority"
    ]
].sort_values(
    "On_Time_Delivery_Pct"
)

# -------------------------------------------------
# STEP 9 — Export supplier analytics outputs
# -------------------------------------------------

output_path = Path("../outputs")

output_path.mkdir(
    parents=True,
    exist_ok=True
)

supplier_scorecard.to_csv(
    output_path / "supplier_scorecard.csv",
    index=False
)

high_risk_suppliers = supplier_scorecard[
    supplier_scorecard["Review_Priority"]
    == "High Review Priority"
].copy()

high_risk_suppliers.to_csv(
    output_path / "high_risk_suppliers.csv",
    index=False
)

print("Supplier analytics exported successfully.")




Suppliers: 8
Purchase order lines: 24
PO_ID                              object
PO_Line                             int64
Supplier_ID                        object
Material_ID                        object
Plant_ID                           object
PO_Creation_Date                   object
Required_Delivery_Date     datetime64[ns]
Confirmed_Delivery_Date            object
Actual_Receipt_Date        datetime64[ns]
Ordered_Qty                         int64
Received_Qty                        int64
Unit_Price_EUR                    float64
PO_Status                          object
Quality_Accepted_Qty                int64
Quality_Rejected_Qty                int64
Actual_Lead_Time_Days               int64
Delivery_Delay_Days                 int64
On_Time_Flag                       object
Open_Qty                            int64
Quality_Rejection_Rate             object
On_Time_Flag_Calc                    bool
dtype: object
  Supplier_ID  Target_On_Time_Rate  Target_On_Time_Pct
0     SUP-0